# Stick Figure Thruster Placement Demo

Interactive UI for placing thrusters on a bilateral stick figure, with physics simulation.

## Features:
- Select body parts to attach thrusters
- Bilateral symmetry: thrusters on left side auto-mirror to right
- Configure thrust direction and force
- Run physics simulation and visualize results

In [ ]:
# ============================================
# COLAB SETUP - Run this cell first!
# ============================================
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Install dependencies
    !pip install -q mujoco ipywidgets matplotlib
    
    # Setup EGL for headless rendering
    !apt-get install -qq libosmesa6-dev libgl1-mesa-glx libglfw3 > /dev/null 2>&1
    
    # Set environment for headless rendering
    import os
    os.environ['MUJOCO_GL'] = 'osmesa'  # Use OSMesa (software rendering) in Colab
    
    # Clone repo if not present
    if not os.path.exists('mobile-suit-sim-mujoco'):
        !git clone https://github.com/OsirisRaptor/mobile-suit-sim-mujoco.git
        os.chdir('mobile-suit-sim-mujoco/notebooks')
    
    print("Colab setup complete!")
else:
    print("Running locally - no special setup needed")

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import mujoco
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

## 1. Body Parts Reference

The stick figure has these attachable body parts:

In [ ]:
# Body part definitions with mirror mappings
BODY_PARTS = {
    # Center line (no mirror)
    "torso": {"mirror": None},
    "head": {"mirror": None},
    "backpack": {"mirror": None},
    # Left side (mirrors to right)
    "left_upper_arm": {"mirror": "right_upper_arm"},
    "left_lower_arm": {"mirror": "right_lower_arm"},
    "left_hand": {"mirror": "right_hand"},
    "left_upper_leg": {"mirror": "right_upper_leg"},
    "left_lower_leg": {"mirror": "right_lower_leg"},
    "left_foot": {"mirror": "right_foot"},
    # Right side (mirrors to left)
    "right_upper_arm": {"mirror": "left_upper_arm"},
    "right_lower_arm": {"mirror": "left_lower_arm"},
    "right_hand": {"mirror": "left_hand"},
    "right_upper_leg": {"mirror": "left_upper_leg"},
    "right_lower_leg": {"mirror": "left_lower_leg"},
    "right_foot": {"mirror": "left_foot"},
}

THRUST_DIRECTIONS = {
    "forward (+X)": (1, 0, 0),
    "backward (-X)": (-1, 0, 0),
    "left (+Y)": (0, 1, 0),
    "right (-Y)": (0, -1, 0),
    "up (+Z)": (0, 0, 1),
    "down (-Z)": (0, 0, -1),
}

print("Body Parts Available:")
print("  Center (no mirror):", [k for k, v in BODY_PARTS.items() if v['mirror'] is None])
print("  Left (mirrors right):", [k for k in BODY_PARTS if k.startswith('left_')])
print("  Right (mirrors left):", [k for k in BODY_PARTS if k.startswith('right_')])

## 2. Thruster Setup Classes

In [ ]:
@dataclass
class ThrusterPlacement:
    """A single thruster attached to a body part."""
    name: str
    body_name: str
    direction: Tuple[float, float, float]  # In body frame
    max_force: float = 1000.0

@dataclass 
class ThrusterSetup:
    """Collection of thrusters with bilateral symmetry support."""
    thrusters: List[ThrusterPlacement] = field(default_factory=list)
    bilateral_symmetry: bool = True
    _counter: int = 0
    
    def add_thruster(self, body_name: str, direction: Tuple[float, float, float], 
                     max_force: float = 1000.0) -> List[ThrusterPlacement]:
        """Add a thruster. Returns list of added thrusters (includes mirror if symmetry on)."""
        added = []
        
        # Primary thruster
        self._counter += 1
        t = ThrusterPlacement(
            name=f"thruster_{self._counter}",
            body_name=body_name,
            direction=direction,
            max_force=max_force,
        )
        self.thrusters.append(t)
        added.append(t)
        
        # Mirror if symmetry enabled and body has a mirror
        if self.bilateral_symmetry and BODY_PARTS.get(body_name, {}).get('mirror'):
            mirror_body = BODY_PARTS[body_name]['mirror']
            # Flip Y for left/right mirroring
            mirror_dir = (direction[0], -direction[1], direction[2])
            
            self._counter += 1
            mt = ThrusterPlacement(
                name=f"thruster_{self._counter}",
                body_name=mirror_body,
                direction=mirror_dir,
                max_force=max_force,
            )
            self.thrusters.append(mt)
            added.append(mt)
        
        return added
    
    def clear(self):
        self.thrusters.clear()
        self._counter = 0
    
    def summary(self) -> str:
        lines = [f"ThrusterSetup ({len(self.thrusters)} thrusters, symmetry={'ON' if self.bilateral_symmetry else 'OFF'})"]
        for t in self.thrusters:
            lines.append(f"  {t.name}: {t.body_name}, dir={t.direction}, {t.max_force}N")
        return "\n".join(lines)

## 3. Interactive Thruster Placement UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create setup
setup = ThrusterSetup(bilateral_symmetry=True)

# Widgets
body_dropdown = widgets.Dropdown(
    options=list(BODY_PARTS.keys()),
    value="backpack",
    description="Body:",
)

direction_dropdown = widgets.Dropdown(
    options=list(THRUST_DIRECTIONS.keys()),
    value="backward (-X)",
    description="Direction:",
)

force_slider = widgets.FloatSlider(
    value=1000,
    min=100,
    max=5000,
    step=100,
    description="Force (N):",
)

symmetry_toggle = widgets.Checkbox(
    value=True,
    description="Bilateral Symmetry (auto-mirror left<->right)",
)

add_btn = widgets.Button(description="Add Thruster", button_style="success")
clear_btn = widgets.Button(description="Clear All", button_style="danger")

output = widgets.Output()

def update_display():
    with output:
        clear_output(wait=True)
        print(setup.summary())

def on_add(b):
    direction = THRUST_DIRECTIONS[direction_dropdown.value]
    added = setup.add_thruster(
        body_name=body_dropdown.value,
        direction=direction,
        max_force=force_slider.value,
    )
    update_display()
    print(f"\nAdded: {[t.name for t in added]}")

def on_clear(b):
    setup.clear()
    update_display()

def on_symmetry(change):
    setup.bilateral_symmetry = change['new']

add_btn.on_click(on_add)
clear_btn.on_click(on_clear)
symmetry_toggle.observe(on_symmetry, names='value')

ui = widgets.VBox([
    widgets.HTML("<h3>Thruster Placement</h3>"),
    widgets.HBox([body_dropdown, direction_dropdown]),
    force_slider,
    symmetry_toggle,
    widgets.HBox([add_btn, clear_btn]),
    widgets.HTML("<hr>"),
    output,
])

update_display()
display(ui)

## 4. Load Stick Figure Model

In [ ]:
# Load the stick figure model
model = mujoco.MjModel.from_xml_path('../models/stick_figure.mjcf')
data = mujoco.MjData(model)

print(f"Model loaded: {model.nbody} bodies, {model.njnt} joints")
print("\nBody names:")
for i in range(model.nbody):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i)
    print(f"  {i}: {name}")

## 5. Run Physics Simulation

In [ ]:
def run_simulation(model, data, setup, n_steps=500, record_interval=10):
    """
    Run physics simulation with thrusters.
    Returns trajectory data for visualization.
    """
    # Reset
    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)
    
    # Build body ID lookup
    body_ids = {}
    for t in setup.thrusters:
        body_ids[t.name] = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, t.body_name)
    
    # Record trajectory
    trajectory = {
        'time': [],
        'com_x': [],
        'com_y': [],
        'com_z': [],
    }
    
    for step in range(n_steps):
        # Clear external forces
        data.xfrc_applied[:] = 0
        
        # Apply thruster forces
        for t in setup.thrusters:
            bid = body_ids[t.name]
            # Get body rotation to transform direction to world frame
            body_xmat = data.xmat[bid].reshape(3, 3)
            direction = np.array(t.direction)
            world_dir = body_xmat @ direction
            world_dir = world_dir / (np.linalg.norm(world_dir) + 1e-8)
            force = world_dir * t.max_force
            data.xfrc_applied[bid, :3] += force
        
        # Step physics
        mujoco.mj_step(model, data)
        
        # Record
        if step % record_interval == 0:
            com = data.subtree_com[1]  # Torso COM
            trajectory['time'].append(data.time)
            trajectory['com_x'].append(com[0])
            trajectory['com_y'].append(com[1])
            trajectory['com_z'].append(com[2])
    
    return trajectory

In [ ]:
# Run simulation with current thruster setup
if len(setup.thrusters) == 0:
    print("No thrusters configured! Add some thrusters above first.")
else:
    print(f"Running simulation with {len(setup.thrusters)} thrusters...")
    print(setup.summary())
    print()
    
    trajectory = run_simulation(model, data, setup, n_steps=500)
    
    # Calculate displacement
    start_pos = np.array([trajectory['com_x'][0], trajectory['com_y'][0], trajectory['com_z'][0]])
    end_pos = np.array([trajectory['com_x'][-1], trajectory['com_y'][-1], trajectory['com_z'][-1]])
    displacement = end_pos - start_pos
    distance = np.linalg.norm(displacement)
    
    print(f"Simulation complete!")
    print(f"  Duration: {trajectory['time'][-1]:.2f} seconds")
    print(f"  Start position: {start_pos}")
    print(f"  End position: {end_pos}")
    print(f"  Displacement: {displacement}")
    print(f"  Total distance: {distance:.2f} m")

## 6. Visualize Trajectory

In [ ]:
if 'trajectory' in dir() and len(trajectory['time']) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # X position over time
    axes[0].plot(trajectory['time'], trajectory['com_x'], 'b-', linewidth=2)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('X Position (m)')
    axes[0].set_title('Forward/Backward Motion')
    axes[0].grid(True, alpha=0.3)
    
    # Y position over time
    axes[1].plot(trajectory['time'], trajectory['com_y'], 'g-', linewidth=2)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Y Position (m)')
    axes[1].set_title('Left/Right Motion')
    axes[1].grid(True, alpha=0.3)
    
    # Z position over time
    axes[2].plot(trajectory['time'], trajectory['com_z'], 'r-', linewidth=2)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Z Position (m)')
    axes[2].set_title('Up/Down Motion')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 3D trajectory
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(trajectory['com_x'], trajectory['com_y'], trajectory['com_z'], 'b-', linewidth=2)
    ax.scatter([trajectory['com_x'][0]], [trajectory['com_y'][0]], [trajectory['com_z'][0]], 
               color='green', s=100, label='Start')
    ax.scatter([trajectory['com_x'][-1]], [trajectory['com_y'][-1]], [trajectory['com_z'][-1]], 
               color='red', s=100, label='End')
    ax.set_xlabel('X (forward)')
    ax.set_ylabel('Y (left)')
    ax.set_zlabel('Z (up)')
    ax.set_title('3D Trajectory')
    ax.legend()
    plt.show()
else:
    print("No trajectory data. Run the simulation cell above first.")

## 7. Example: Balanced vs Unbalanced Thrusters

Compare how bilateral symmetry affects motion:

In [ ]:
# Test 1: Symmetric hand thrusters (should mostly cancel)
setup_symmetric = ThrusterSetup(bilateral_symmetry=True)
setup_symmetric.add_thruster('left_hand', direction=(0, -1, 0), max_force=1000)  # Push outward
print("Symmetric setup:")
print(setup_symmetric.summary())

traj_sym = run_simulation(model, data, setup_symmetric, n_steps=300)
disp_sym = np.sqrt(
    (traj_sym['com_x'][-1] - traj_sym['com_x'][0])**2 +
    (traj_sym['com_y'][-1] - traj_sym['com_y'][0])**2 +
    (traj_sym['com_z'][-1] - traj_sym['com_z'][0])**2
)
print(f"Total displacement: {disp_sym:.3f} m\n")

# Test 2: Single backpack thruster (no symmetry, clear directional movement)
setup_single = ThrusterSetup(bilateral_symmetry=False)
setup_single.add_thruster('backpack', direction=(1, 0, 0), max_force=2000)  # Push forward
print("Single thruster setup:")
print(setup_single.summary())

traj_single = run_simulation(model, data, setup_single, n_steps=300)
disp_single = np.sqrt(
    (traj_single['com_x'][-1] - traj_single['com_x'][0])**2 +
    (traj_single['com_y'][-1] - traj_single['com_y'][0])**2 +
    (traj_single['com_z'][-1] - traj_single['com_z'][0])**2
)
print(f"Total displacement: {disp_single:.3f} m")

## 8. Export Thruster Configuration

Export your current setup for use in other simulations:

In [ ]:
import json

# Export current setup as JSON
export_data = {
    'bilateral_symmetry': setup.bilateral_symmetry,
    'thrusters': [
        {
            'name': t.name,
            'body_name': t.body_name,
            'direction': list(t.direction),
            'max_force': t.max_force,
        }
        for t in setup.thrusters
    ]
}

print("Exported Configuration:")
print(json.dumps(export_data, indent=2))

---

## Summary

This notebook demonstrates:

1. **Thruster Placement UI** - Select body part, direction, force
2. **Bilateral Symmetry** - Auto-mirror thrusters left<->right
3. **Physics Simulation** - MuJoCo simulates forces and limb dynamics
4. **Trajectory Visualization** - See how the figure moves in 3D space

### Next Steps:
- Add rendering to generate GIFs of the simulation
- Implement thruster scheduling (on/off over time)
- Add AMBAC control (limb movement for attitude control)

## 9. Render Simulation (Colab Compatible)

Generate a GIF of the simulation. Uses OSMesa for software rendering in Colab.

In [ ]:
def render_simulation(model, data, setup, n_steps=300, fps=30, width=640, height=480):
    """
    Render simulation to frames. Works in Colab with OSMesa.
    Returns list of RGB frames.
    """
    import mujoco
    
    # Reset simulation
    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)
    
    # Build body ID lookup
    body_ids = {}
    for t in setup.thrusters:
        body_ids[t.name] = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, t.body_name)
    
    # Create renderer
    try:
        renderer = mujoco.Renderer(model, width, height)
    except Exception as e:
        print(f"Renderer creation failed: {e}")
        print("Tip: In Colab, make sure the setup cell ran successfully")
        return None
    
    frames = []
    step_per_frame = max(1, int(1.0 / (model.opt.timestep * fps)))
    
    for step in range(n_steps):
        # Apply thruster forces
        data.xfrc_applied[:] = 0
        for t in setup.thrusters:
            bid = body_ids[t.name]
            body_xmat = data.xmat[bid].reshape(3, 3)
            direction = np.array(t.direction)
            world_dir = body_xmat @ direction
            world_dir = world_dir / (np.linalg.norm(world_dir) + 1e-8)
            force = world_dir * t.max_force
            data.xfrc_applied[bid, :3] += force
        
        # Step physics
        mujoco.mj_step(model, data)
        
        # Render frame
        if step % step_per_frame == 0:
            renderer.update_scene(data, camera='fixed')
            frame = renderer.render()
            frames.append(frame.copy())
    
    renderer.close()
    return frames

# Only run if thrusters are configured
if len(setup.thrusters) > 0:
    print(f"Rendering simulation with {len(setup.thrusters)} thrusters...")
    frames = render_simulation(model, data, setup, n_steps=300)
    
    if frames:
        print(f"Rendered {len(frames)} frames")
        
        # Display first and last frame
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(frames[0])
        axes[0].set_title("Start")
        axes[0].axis('off')
        axes[1].imshow(frames[-1])
        axes[1].set_title("End")
        axes[1].axis('off')
        plt.show()
else:
    print("Add thrusters first using the UI above!")

In [ ]:
# Save as GIF and display inline
if 'frames' in dir() and frames:
    try:
        import imageio
    except ImportError:
        !pip install -q imageio
        import imageio
    
    gif_path = 'thruster_sim.gif'
    imageio.mimsave(gif_path, frames, fps=30)
    print(f"Saved to {gif_path}")
    
    # Display in notebook
    from IPython.display import Image, display
    display(Image(filename=gif_path))